In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.6 Symplectic vs. Naive Integrators

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.6",
    title="Symplectic vs. Naive Integrators",
    blurb="Why velocity-Verlet keeps energy bounded for millions of steps "
    "while Euler and even RK4 drift: order of accuracy, phase-space "
    "area, and the geometry behind a good integrator.",
    difficulty="intermediate",
    estimate="60–90 min",
)

## Notebook overview

Throughout Volume I we used energy conservation as a *check* on our integrators:
the double pendulum ([§1.3](double-pendulum.ipynb)), the Kepler orbit
([§1.4](kepler-orbits.ipynb)), and the coupled chain ([§1.5](coupled-oscillators.ipynb))
were all certified by watching their total energy hold steady to ten or eleven
digits. We trusted those checks. This capstone turns the running theme into the
subject itself and asks the question underneath it: *why* do some integrators
deserve that trust over long times while others (even very accurate ones) do
not?

The answer is geometric. A good long-time integrator is one that preserves the
right structure of the flow, not merely the one with the smallest per-step
error. We will see that the humble **velocity-Verlet** scheme, only
second-order accurate, keeps the energy of a harmonic oscillator bounded for
millions of steps, while the fourth-order **Runge–Kutta (RK4)** method (far
more accurate per step) slowly bleeds energy away. The distinguishing property
is **symplecticity**: preservation of phase-space area.

We will (1) implement four steppers, (2) run the headline energy comparison over
many periods, (3) show Euler's energy grows by exactly $(1+h^2)$ per step, (4)
show Verlet's energy error is *bounded*, not merely small, (5) measure each
method's order of accuracy, (6) animate a blob of initial conditions to *see*
phase-space area preserved or destroyed, and (7) apply the methods to a Kepler
orbit, where a symplectic step keeps the ellipse closed while Euler spirals out.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the
> mechanics see Nolting, *Theoretical Physics 1* {cite}`nolting1`; for the full
> geometric-integration picture (symplectic maps, modified shadow
> Hamiltonians, backward error analysis), see Hairer, Lubich & Wanner,
> *Geometric Numerical Integration* {cite}`hairer-gni`.

## Theory in brief

### The test system: the harmonic oscillator

We test every method on the system whose exact answer we know completely: the
unit harmonic oscillator $\dot q = p$, $\dot p = -q$ (mass $m=1$, frequency
$\omega=1$). Its energy is

```{math}
:label: eq-ho-energy
E = \tfrac12\left(q^2 + p^2\right),
```

the exact period is $2\pi$, and the exact trajectory is a *rotation* of the
phase-space point $(q,p)$ at unit angular rate: a circle of constant radius
$\sqrt{2E}$. Any drift in that radius is purely the integrator's fault, which is
what makes this system the perfect proving ground.

A one-step integrator advances $(q_n, p_n) \mapsto (q_{n+1}, p_{n+1})$ by a
fixed step $h$. For a linear system each step is a matrix; the **determinant of
that matrix** is the factor by which it scales phase-space area, and over long
times it governs the energy trend as well: secular growth if the determinant
exceeds 1, no net growth if it is exactly 1.

### Explicit (forward) Euler

The most naive scheme evaluates the slope at the *old* point:

```{math}
:label: eq-euler
q_{n+1} = q_n + h\,p_n, \qquad p_{n+1} = p_n - h\,q_n .
```

Its step matrix is $\bigl(\begin{smallmatrix} 1 & h \\ -h & 1
\end{smallmatrix}\bigr)$, with determinant $1 + h^2 > 1$. Every step inflates
phase-space area (and energy) by the factor $1+h^2$, so after $n$ steps the
energy is $(1+h^2)^n E_0$: an exponential blow-up, no matter how small $h$ is.
Forward Euler is **first-order accurate** ($O(h)$ global error) and *not*
symplectic.

### Symplectic (semi-implicit) Euler

A one-character change rescues it: update $p$ first, then use the *new* $p$ to
update $q$,

```{math}
:label: eq-symp
p_{n+1} = p_n - h\,q_n, \qquad q_{n+1} = q_n + h\,p_{n+1} .
```

The step matrix is now $\bigl(\begin{smallmatrix} 1-h^2 & h \\ -h & 1
\end{smallmatrix}\bigr)$, with determinant **exactly 1**. The method is
area-preserving (*symplectic*), though still only first-order accurate. It does
not conserve $E$ exactly, but it conserves a nearby *modified* energy, so the
true energy merely oscillates within a bounded band.

### Velocity Verlet

The workhorse of molecular dynamics splits each step into a half-kick, a full
drift, and a half-kick:

```{math}
:label: eq-verlet
p_{n+\frac12} = p_n - \tfrac{h}{2}\,q_n, \qquad
q_{n+1} = q_n + h\,p_{n+\frac12}, \qquad
p_{n+1} = p_{n+\frac12} - \tfrac{h}{2}\,q_{n+1} .
```

Velocity Verlet is **second-order accurate** and symplectic (step determinant
exactly 1). Crucially it conserves a *shadow* Hamiltonian that differs from the
true $E$ by $O(h^2)$, so the true energy stays bounded and *oscillatory* with no
secular drift: bounded forever, not just initially small.

### Runge–Kutta 4

The classic fourth-order scheme combines four slope evaluations:

```{math}
:label: eq-rk4
(q,p)_{n+1} = (q,p)_n + \tfrac{h}{6}\bigl(\mathbf k_1 + 2\mathbf k_2
  + 2\mathbf k_3 + \mathbf k_4\bigr),
```

with $\mathbf k_1 = f(\,\cdot_n\,)$, $\mathbf k_2 = f(\,\cdot_n +
\tfrac{h}{2}\mathbf k_1)$, $\mathbf k_3 = f(\,\cdot_n + \tfrac{h}{2}\mathbf
k_2)$, $\mathbf k_4 = f(\,\cdot_n + h\,\mathbf k_3)$ and $f(q,p) = (p,-q)$. RK4
has a tiny per-step error ($O(h^5)$) but is **not symplectic**: its step
determinant differs from 1 at $O(h^6)$, so over very long runs the energy drifts
*secularly*: slowly but without bound.

### The lesson

For long-time dynamics the *geometry* a method preserves can matter more than
its order of accuracy. A symplectic method (Verlet, symplectic Euler) trades a
little per-step accuracy for a structural guarantee (bounded energy) that a
higher-order non-symplectic method (RK4) cannot offer. That is why the energy
checks we trusted in [§1.3](double-pendulum.ipynb)–[§1.5](coupled-oscillators.ipynb)
were trustworthy *there* (an adaptive high-order
solver over a modest number of periods) but would be the wrong tool for a
billion-step molecular-dynamics run.

---
## Setup

We implement four steppers, each with the identical signature
`(q, p, h) -> (q, p)`, and a `run(stepper, h, n)` driver that returns the
trajectories `Q`, `P` and the energy series `E`. The default force is the
harmonic oscillator; a general `force` callback lets Exercise 7 reuse the same
Verlet/Euler kernels on the Kepler $1/r^2$ field from [§1.4](kepler-orbits.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.animate import show


# --- The four steppers for the harmonic oscillator q'' = -q -------------------
def step_euler(q, p, h):
    """One explicit (forward) Euler step.

    Uses the slope at the OLD point; its energy drifts upward, the cautionary baseline.

    Parameters
    ----------
    q, p : float
        Current position and momentum.
    h : float
        Time step.

    Returns
    -------
    tuple of float
        The updated ``(q, p)``.
    """
    return q + h * p, p - h * q


def step_symplectic_euler(q, p, h):
    """One semi-implicit (symplectic) Euler step.

    Updates p first, then q with the NEW p; symplectic, so energy stays bounded.

    Parameters
    ----------
    q, p : float
        Current state.
    h : float
        Time step.

    Returns
    -------
    tuple of float
        The updated ``(q, p)``.
    """
    p_new = p - h * q
    return q + h * p_new, p_new


def step_verlet(q, p, h):
    """One velocity-Verlet step (half-kick, drift, half-kick).

    The workhorse symplectic integrator: second order with bounded energy error.

    Parameters
    ----------
    q, p : float
        Current state.
    h : float
        Time step.

    Returns
    -------
    tuple of float
        The updated ``(q, p)``.
    """
    p_half = p - 0.5 * h * q
    q_new = q + h * p_half
    return q_new, p_half - 0.5 * h * q_new


def _f(q, p):
    """Harmonic-oscillator slope (q', p') = (p, −q)."""
    return p, -q


def step_rk4(q, p, h):
    """One classic fourth-order Runge-Kutta step on the oscillator.

    High local accuracy but not symplectic, so its energy slowly drifts over long runs.

    Parameters
    ----------
    q, p : float
        Current state.
    h : float
        Time step.

    Returns
    -------
    tuple of float
        The updated ``(q, p)``.
    """
    k1q, k1p = _f(q, p)
    k2q, k2p = _f(q + 0.5 * h * k1q, p + 0.5 * h * k1p)
    k3q, k3p = _f(q + 0.5 * h * k2q, p + 0.5 * h * k2p)
    k4q, k4p = _f(q + h * k3q, p + h * k3p)
    q_new = q + h / 6.0 * (k1q + 2 * k2q + 2 * k3q + k4q)
    p_new = p + h / 6.0 * (k1p + 2 * k2p + 2 * k3p + k4p)
    return q_new, p_new


def run(stepper, h, n, q0=1.0, p0=0.0):
    """Iterate a stepper and record the trajectory and energy.

    Parameters
    ----------
    stepper : callable
        A one-step integrator ``(q, p, h) -> (q, p)``.
    h : float
        Time step.
    n : int
        Number of steps.
    q0, p0 : float, optional
        Initial state.

    Returns
    -------
    Q, P, E : numpy.ndarray
        Position, momentum, and energy histories.
    """
    Q = np.empty(n + 1)
    P = np.empty(n + 1)
    Q[0], P[0] = q0, p0
    q, p = q0, p0
    for i in range(n):
        q, p = stepper(q, p, h)
        Q[i + 1], P[i + 1] = q, p
    E = 0.5 * (Q**2 + P**2)
    return Q, P, E


PERIOD = 2.0 * np.pi  # exact period of the unit oscillator

## Exercise 1 — Implement the four steppers

The four update rules of the theory section (forward Euler {eq}`eq-euler`,
symplectic Euler {eq}`eq-symp`, velocity Verlet {eq}`eq-verlet`, and RK4
{eq}`eq-rk4`) are each a map $(q,p) \mapsto (q,p)$. The cleanest way to be sure
they are transcribed correctly is to take a single step by hand from a known
starting point and compare.

With the steppers defined in Setup, take **one** step of size $h=0.1$ from
$(q,p)=(1,0)$ with each method and check it against the value you get by
substituting into the formulas by hand.

1. Forward Euler {eq}`eq-euler`: expect $(1,\,-0.1)$.
2. Symplectic Euler {eq}`eq-symp`: expect $(1-0.1\cdot0.1,\,-0.1) =
   (0.99,\,-0.1)$.
3. Velocity Verlet {eq}`eq-verlet`: expect $(0.995,\,-0.0997\ldots)$.
4. RK4 {eq}`eq-rk4`: expect the rotation-by-$h$ value to $O(h^5)$.

**Write this one yourself** — the implementation is the lesson.

In [ ]:
# (solution hidden on the public site)


In [ ]:
for name in one_step:
    validate.close(
        np.array(one_step[name]),
        np.array(expected[name]),
        f"one step of {name} matches the hand computation",
        rtol=1e-12,
    )

## Exercise 2 — Energy over many periods: the headline comparison

This is the centrepiece. Run all four methods on the oscillator at a *fixed*
step $h=0.05$ for about sixteen periods and watch the energy {eq}`eq-ho-energy`.
The theory makes sharp predictions: forward Euler {eq}`eq-euler` should blow up
geometrically as $(1+h^2)^n$, RK4 {eq}`eq-rk4` should sag almost imperceptibly,
and the two symplectic methods should stay essentially flat.

1. Run each of the four steppers (forward Euler, symplectic Euler, velocity
   Verlet, RK4) at $h=0.05$ for $n$ steps covering $\approx 16$ periods, and
   compute $E(t)/E_0$ for each with the `run` driver.
2. Plot all four energy ratios on one axes (a log $y$-scale shows the
   four-decade spread). This single figure is the whole notebook in a picture.
3. Confirm Euler's energy has blown up (final ratio $> 10$) while Verlet's
   stays bounded ($\max|E/E_0 - 1| < 10^{-3}$).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    E_euler[-1] / E_euler[0] > 10,
    "explicit Euler energy blows up over many periods",
    f"final E/E0 = {E_euler[-1] / E_euler[0]:.1f}",
)

In [ ]:
validate.check(
    np.max(np.abs(E_verlet / E_verlet[0] - 1)) < 1e-3,
    "velocity-Verlet energy stays bounded",
    f"max|E/E0-1| = {np.max(np.abs(E_verlet / E_verlet[0] - 1)):.2e}",
)

## Exercise 3 — Euler's geometric energy growth

The blow-up in Exercise 2 is not vague numerical noise; it is an exact, knowable
rate. The step matrix of forward Euler {eq}`eq-euler` has determinant $1+h^2$,
and for this oscillator the energy {eq}`eq-ho-energy` scales with phase-space
area, so **each step multiplies the energy by exactly $1+h^2$**.

**Your task.** Take a single Euler step from $(1,0)$ and confirm the ratio
$E_1/E_0$ equals $1+h^2$ to machine precision. (Iterating this is the
$(1+h^2)^n$ exponential we just saw.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    E3[1] / E3[0],
    1 + H3**2,
    "Euler grows energy by exactly (1+h²) per step",
    rtol=1e-6,
)

## Exercise 4 — Verlet energy is *bounded*, not just small

A method could have a small energy error that nonetheless creeps in one
direction forever: that is *secular drift*, and it is what eventually ruins a
long run. The symplectic guarantee is stronger than "small error": it says the
energy error of velocity Verlet {eq}`eq-verlet` is *bounded*, oscillating within
a fixed band that does not grow with time. RK4 {eq}`eq-rk4`, by contrast, has a
far smaller error that nevertheless drifts.

1. Run velocity Verlet for a long run ($\sim 20\,000$ steps) and compare
   the maximum relative energy deviation in the **first half** of the run to
   that in the **second half** (`numpy.max` on each half). For a bounded
   (non-drifting) method they are equal.
2. Run RK4 over the same span and plot both energy series, so you can see
   Verlet's steady ripple against RK4's slow downward sag.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    maxdev_second,
    maxdev_first,
    "Verlet energy error does not grow with time (bounded, not secular)",
    rtol=0.1,
)

## Exercise 5 — Order of accuracy

Symplecticity governs *long-time* behaviour; the **order of accuracy** governs
the *per-step* error and how fast it shrinks as $h\to 0$. Euler {eq}`eq-euler`
is first order, Verlet {eq}`eq-verlet` second, RK4 {eq}`eq-rk4` fourth. We
measure each by computing the global error $\lvert(q,p)-(q,p)_\text{exact}\rvert$
at a fixed final time over a range of step sizes and fitting the slope on a
log–log plot.

> **Measure at a generic time.** Evaluate the error at $T=10$, *not* at a
> multiple of the period $2\pi$. At $T=2\pi n$ the oscillator's global error has
> a node where the leading error term vanishes, and the fitted order collapses
> to nonsense. This is a real subtlety, not a bug in the code: a ✗ here may
> just mean the measurement was at the wrong $T$.

**Your task.** For forward Euler, velocity Verlet, and RK4, integrate to
$T=10$ at several step sizes, measure the error against the exact rotation
$(\cos T, -\sin T)$, and fit the convergence order as the log–log slope
(`numpy.polyfit` on the logs). Expect $\approx 1, 2, 4$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    fitted_order["forward Euler"], 1.0, "forward Euler is first order", atol=0.15
)

In [ ]:
validate.close(
    fitted_order["velocity Verlet"], 2.0, "velocity Verlet is second order", atol=0.15
)

In [ ]:
validate.close(fitted_order["RK4"], 4.0, "RK4 is fourth order", atol=0.2)

## Exercise 6 — Phase-space area: the geometry behind it all (worked animation)

Here is *why* symplectic methods conserve energy, made visible. Liouville's
theorem says the exact flow of a Hamiltonian system preserves phase-space area:
a blob of initial conditions may distort in shape but never changes its area.
An integrator that respects this (a symplectic one) keeps energy bounded; one
that violates it does not. We take a small ring of initial conditions in
$(q,p)$ and evolve every point with Euler {eq}`eq-euler` and, separately, with
Verlet {eq}`eq-verlet`.

The animation overlays the two evolving blobs. Euler's inflates without bound
(its area grows as $(1+h^2)^n$, exactly the energy factor of Exercise 3), while
Verlet's rotates rigidly with its area fixed. This is Liouville's theorem
violated vs. respected. This is the worked example for the two-animation rule;
you build the second one in Exercise 7.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    area_verlet,
    "Verlet preserves phase-space area (symplectic)",
    rel_drift=1e-3,
)

In [ ]:
validate.check(
    area_euler[-1] > 1.5 * area_euler[0],
    "Euler inflates phase-space area (violates Liouville)",
    f"area grew ×{area_euler[-1] / area_euler[0]:.2f}",
)

## Exercise 7 — Long-time orbit fidelity (student-implemented animation)

The payoff for real physics: take the gravitational $1/r^2$ force from
[§1.4](kepler-orbits.ipynb) and integrate a bound elliptical orbit for many periods. The exact orbit is
a *closed* ellipse: it should retrace itself forever. A symplectic step
(velocity Verlet) keeps it closed; forward Euler, injecting energy every step,
spirals the planet steadily outward. This is the same energy story as the
oscillator, now with a recognisable orbit attached.

We reuse the inverse-square acceleration $\ddot{\mathbf r} = -\mathbf r/r^3$ and
the bound initial condition of [§1.4](kepler-orbits.ipynb) ($\mathbf r_0=(1,0)$, $\mathbf v_0=(0,1.2)$,
in units $GM=1$). The energy is $E=\tfrac12 v^2 - 1/r$.

1. Integrate the orbit for many periods with Euler and (separately) with
   velocity Verlet, both at the same small step (data prepared below).
2. **Build the animation** comparing the two orbits side by side: you have
   the trajectories `(xe, ye)` and `(xv, yv)`; assemble a `FuncAnimation` that
   traces both paths, `plt.close(fig)`, then display with `ecp.animate.show`.
   Target look: Verlet a clean closed ellipse, Euler a widening spiral.
3. Quantify the story: confirm Verlet's energy stays bounded over the run
   while Euler's energy rises systematically (it *injects* energy).

> A ✗ on the final checks is about the **energy series we computed**, not the
> animation: there are many valid ways to draw two orbits. If a check fails,
> inspect `E_verlet` and `E_euler`: the physics lives there, and any correct
> `FuncAnimation` drawing the same data is fine.

In [ ]:
# (solution hidden on the public site)


**Part 3 — quantify.** The animated difference is backed by the energy series: Verlet
holds the orbit's energy in a bounded band over all twenty periods, while
Euler's energy climbs steadily: each step adds a little, exactly as it inflated
the oscillator's energy in Exercise 2. (For a *bound* orbit $E_0<0$, "injecting
energy" means $E$ rises *toward* zero, so we track the rise relative to the
initial magnitude $|E_0|$.)

In [ ]:
validate.conserved(
    E_verlet_orb,
    "Verlet keeps the orbit's energy bounded over many periods",
    rel_drift=1e-2,
)

In [ ]:
energy_rise = (E_euler_orb[-1] - E_euler_orb[0]) / abs(E_euler_orb[0])
validate.check(
    energy_rise > 0.1,
    "Euler systematically injects energy into the orbit",
    f"energy rose by {energy_rise:+.2f} of |E0| over {N_ORB} periods",
)

## Notebook summary

- Four steppers (forward Euler, symplectic Euler, velocity Verlet, RK4) compared on the same
  oscillator: Euler's energy grows geometrically, Verlet's energy stays **bounded** (not just
  small), and each method's order of accuracy is measured from its error scaling.
- The geometry behind it: symplectic integrators preserve phase-space area, which is why Verlet
  keeps long-time orbits faithful where RK4's energy slowly drifts.

## Outlook

- **Verlet on the nonlinear pendulum ([§1.2](damped-driven-pendulum.ipynb)) and
  the double pendulum ([§1.3](double-pendulum.ipynb)).**
  Replace the adaptive solver used there with velocity Verlet and compare the
  long-time energy behaviour: the bounded ripple is exactly the guarantee that
  made those energy checks meaningful.
- **Higher-order symplectic schemes.** The Forest–Ruth and Yoshida composition
  methods reach fourth order *while staying symplectic*; implement one and
  confirm its order with the Exercise 5 machinery, then watch its energy stay
  bounded where RK4 drifts.
- **Why molecular dynamics uses velocity Verlet.** Bounded energy over billions
  of steps, time-reversibility, and one force evaluation per step make it the
  default for the statistical-mechanics simulations of Volume VI: the bridge
  from this capstone to that volume.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()